# A1: Train và evaluate 6 mô hình trên cùng GPU Colab

Mở **Runtime → Change runtime type → GPU**, rồi chạy các ô từ trên xuống. GPU thực tế sẽ được ghi lại; không cần chọn một loại GPU cụ thể.

Bản có theo dõi tiến trình: 6 ô model riêng, bảng trạng thái trực tiếp, batch/epoch, loss/accuracy, ETA theo model và log lỗi.

Notebook dùng code từ nhánh **a1/m2**, giữ cấu hình 50 epoch / seed 42 / batch 128 / Adam lr 0.001. Dữ liệu và checkpoint đang train nằm trên ổ đĩa Colab để giảm ảnh hưởng của I/O Drive đến thời gian train. Sau mỗi bước train/evaluate, toàn bộ kết quả được sao lưu vào `MyDrive/BTL_DL_Result/VanhResult/a1_v1/`. File trùng tên trong thư mục này sẽ được ghi đè.

Nếu runtime bị ngắt giữa lúc train một model, phần chưa sao lưu của model đó có thể mất. Script hiện tại chưa hỗ trợ tự động resume giữa các epoch. Các model đã hoàn tất được sao lưu trên Drive.

Cấu trúc output trùng với `outputs/a1_v1/` trên local. Nhánh `a1/m2` cần chứa PR cập nhật cấu trúc này trước khi chạy.


In [ ]:
import torch

assert torch.cuda.is_available(), "Chưa có CUDA. Hãy chọn GPU trong Runtime → Change runtime type."
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

## 1. Chuẩn bị code và thư viện
Giữ bản torch/torchvision CUDA được Colab cung cấp. Dùng Python của runtime cho cả 6 model.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torchvision

requirements = [
    "loguru>=0.7,<1",
    "matplotlib>=3.8,<4",
    "numpy>=1.26,<3",
    "pandas>=2.2,<3",
    "pyyaml>=6.0,<7",
    "scikit-learn>=1.4,<2",
    "seaborn>=0.13,<1",
    "tqdm>=4.66,<5",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *requirements], check=True)

repo_dir = Path("/content/CO3133-Deep-Learning-261-Placeholder")
if not repo_dir.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            "a1/m2",
            "--single-branch",
            "https://github.com/vietanh1802/CO3133-Deep-Learning-261-Placeholder.git",
            str(repo_dir),
        ],
        check=True,
    )
os.chdir(repo_dir)
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Source commit:", revision)

assert torch.cuda.is_available(), "CUDA không còn khả dụng."
print("torchvision:", torchvision.__version__)

# This notebook requires the outputs/a1_v1 layout in the selected branch.
training_source = (repo_dir / "scripts/training/train_image_classification.py").read_text()
assert 'Path(config.results_dir) / "a1"' not in training_source, (
    "Hãy merge PR cấu trúc outputs vào a1/m2 trước khi chạy notebook này."
)

## 2. Kết nối Drive và chuẩn bị đầu ra
Chấp thuận kết nối Drive khi Colab hỏi. Output được sao lưu trực tiếp vào `MyDrive/BTL_DL_Result/VanhResult/a1_v1/`, với các thư mục con `checkpoints/` và `results/`; không thêm thư mục `run_id`.

Mỗi lần chạy ô này tạo vùng làm việc mới trên đĩa tạm Colab, nhưng dùng cùng thư mục Drive. File trùng tên trên Drive sẽ được ghi đè khi sao lưu. Đừng chạy lại ô này giữa một đợt train.


In [ ]:
import json
import platform
import shutil
import tempfile
from datetime import datetime
from zoneinfo import ZoneInfo

import yaml
from google.colab import drive  # ty: ignore[unresolved-import]

drive.mount("/content/drive")
run_dir = Path(tempfile.mkdtemp(prefix="a1_v1_", dir="/content"))
saved_dir = Path("/content/drive/MyDrive/BTL_DL_Result/VanhResult/a1_v1")
saved_dir.mkdir(parents=True, exist_ok=True)
config_dir = run_dir / "configs"
config_dir.mkdir()
models = ["linear", "mlp", "cnn", "gru", "lstm", "transformer"]

configs = {}
for model_name in models:
    source = repo_dir / "configs/image_classification/fashion_mnist" / f"{model_name}.yaml"
    config = yaml.safe_load(source.read_text())
    config["dataset"] = "fashion_mnist"
    config["data_dir"] = str(repo_dir / "data")
    config["checkpoint_dir"] = str(run_dir / "checkpoints")
    config["results_dir"] = str(run_dir / "results")
    (config_dir / f"{model_name}.yaml").write_text(yaml.safe_dump(config, sort_keys=False))
    configs[model_name] = config

common = ["seed", "batch_size", "learning_rate", "epochs", "num_workers", "val_fraction", "dataset"]
for key in common:
    assert len({configs[name][key] for name in models}) == 1, f"Khác thiết lập: {key}"
assert configs["lstm"]["epochs"] == 50

environment = {
    "experiment": "a1_v1",
    "created_at": datetime.now(ZoneInfo("Asia/Bangkok")).isoformat(),
    "source_branch": "a1/m2",
    "source_commit": revision,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_build": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0),
    "gpu_memory_bytes": torch.cuda.get_device_properties(0).total_memory,
    "common_config": {key: configs["lstm"][key] for key in common},
    "timing_scope": "Training includes training, validation, and checkpoint writes on Colab local disk. Inference times model forward passes only.",
}
(run_dir / "environment.json").write_text(json.dumps(environment, indent=2))
(run_dir / "nvidia-smi.txt").write_text(subprocess.check_output(["nvidia-smi"], text=True))
(run_dir / "requirements-freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
)


def output_files():
    for source in run_dir.rglob("*"):
        if source.is_file():
            yield source, source.relative_to(run_dir)


def backup_outputs():
    for source, relative in output_files():
        destination = saved_dir / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)


backup_outputs()
print("Local run:", run_dir)
print("Drive backup:", saved_dir)
print("Shared settings:", environment["common_config"])

## 3. Theo dõi tiến trình trực tiếp

Chạy ô **3.0** một lần, rồi chạy từng ô model bên dưới hoặc dùng **Run all** cho một đợt mới. Mỗi model có bảng trạng thái cả 6 model; batch, epoch, loss, accuracy và thời gian được cập nhật trực tiếp. Log đầy đủ nằm ở `console/` và được sao lưu cùng output.

**Đổi từ notebook cũ trong cùng runtime:** giữ nguyên các biến `run_dir`, `saved_dir`, `configs`, `config_dir`, `models`, `environment` và `backup_outputs`. Chỉ thêm/chạy ô 3.0 và các ô model mới; không chạy lại ô tạo thư mục ở phần 2. Chờ ô train cũ kết thúc hoặc dừng nó trước, tránh chạy hai tiến trình train cùng lúc. Dừng giữa một model sẽ cần train lại model đó từ đầu; các model đã hoàn tất sẽ được bỏ qua nếu file khớp.


In [ ]:
# 3.0 — Khởi tạo bảng theo dõi (chạy một lần)
import html
import queue
import re
import threading
import time
from collections import deque
from typing import Any

from IPython.display import HTML, clear_output, display  # ty: ignore[unresolved-import]

ANSI = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")
EPOCH_LOG = re.compile(
    r"Epoch\s+(\d+)/(\d+)\s*\|\s*train loss\s+([-+\d.eE]+),\s*accuracy\s+([-+\d.eE]+)"
    r"\s*\|\s*validation loss\s+([-+\d.eE]+),\s*accuracy\s+([-+\d.eE]+)"
)
BATCH_LOG = re.compile(r"(Training|Validating):.*?(\d+)\s*/\s*(\d+)")


def read_json(path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return None


def training_complete(model_name):
    config = configs[model_name]
    result_dir = run_dir / "results" / config["name"]
    summary = read_json(result_dir / "training_summary.json")
    checkpoint = run_dir / "checkpoints" / config["name"] / "best.pt"
    return bool(
        summary
        and checkpoint.exists()
        and summary.get("config") == config
        and summary.get("epochs") == config["epochs"]
        and len(summary.get("train_loss", [])) == config["epochs"]
        and len(summary.get("validation_loss", [])) == config["epochs"]
    )


def evaluation_complete(model_name):
    config = configs[model_name]
    result_dir = run_dir / "results" / config["name"]
    summary = read_json(result_dir / "training_summary.json")
    record = read_json(result_dir / "test_result.json")
    return bool(
        summary
        and record
        and record.get("config") == config
        and record.get("sample_count") == 10000
        and record.get("split") == "test"
        and record.get("checkpoint", {}).get("epoch") == summary.get("best_epoch")
        and record.get("checkpoint", {}).get("best_validation_loss")
        == summary.get("best_validation_loss")
    )


def duration(seconds):
    if seconds is None:
        return "—"
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


def number(value):
    return "—" if value is None else f"{value:.4f}"


def percent(value):
    return "—" if value is None else f"{100 * value:.2f}%"


class A1Progress:
    LABELS = {
        "pending": "Chờ chạy",
        "train": "Đang train",
        "validation": "Validation",
        "evaluate": "Đang test",
        "backup": "Sao lưu Drive",
        "ready": "Chờ test / sao lưu",
        "done": "Hoàn tất",
        "error": "Có lỗi",
        "stopped": "Đã dừng",
    }

    def __init__(self):
        self.rows: dict[str, dict[str, Any]] = {}
        self.active: str | None = None
        self.started = time.monotonic()
        self.stage_started = self.started
        self.last_render = 0.0
        self.tail = deque(maxlen=6)
        self.last_output = None
        self.console_path = None
        for name in models:
            self.rows[name] = {
                "stage": "pending",
                "epoch": 0,
                "epochs": configs[name]["epochs"],
                "train_loss": None,
                "val_loss": None,
                "train_acc": None,
                "val_acc": None,
                "best_val": None,
                "test_acc": None,
                "test_f1": None,
                "batch": None,
                "batch_total": None,
                "detail": "",
                "epoch_durations": [],
                "last_epoch_at": None,
                "train_seconds": None,
            }
            if training_complete(name):
                self.apply_summary(name)
                self.rows[name]["stage"] = "ready"
                if evaluation_complete(name):
                    record = read_json(
                        run_dir / "results" / configs[name]["name"] / "test_result.json"
                    )
                    self.rows[name].update(
                        test_acc=record["metrics"]["accuracy"],
                        test_f1=record["metrics"]["macro_f1"],
                    )
                    saved_result = (
                        saved_dir / "results" / configs[name]["name"] / "test_result.json"
                    )
                    if saved_result.exists():
                        self.rows[name]["stage"] = "done"

    def apply_summary(self, name):
        summary = read_json(run_dir / "results" / configs[name]["name"] / "training_summary.json")
        row = self.rows[name]
        row.update(
            epoch=summary["epochs"],
            train_loss=summary["train_loss"][-1],
            val_loss=summary["validation_loss"][-1],
            best_val=summary["best_validation_loss"],
            train_seconds=summary["training_seconds"],
            batch=None,
            batch_total=None,
        )
        log_path = run_dir / "console" / configs[name]["name"] / "train.log"
        if log_path.exists():
            matches = list(
                EPOCH_LOG.finditer(log_path.read_text(encoding="utf-8", errors="replace"))
            )
            if matches and int(matches[-1][1]) == summary["epochs"]:
                row.update(train_acc=float(matches[-1][4]), val_acc=float(matches[-1][6]))

    def stage(self, name, stage, detail=""):
        self.active = name
        self.rows[name].update(stage=stage, detail=detail, batch=None, batch_total=None)
        self.stage_started = time.monotonic()
        self.render(force=True)

    def consume(self, name, raw, mode):
        line = ANSI.sub("", raw).strip()
        if not line:
            return
        self.last_output = time.monotonic()
        self.tail.append(line[-700:])
        row = self.rows[name]
        match = EPOCH_LOG.search(line)
        if mode == "train" and match:
            epoch, total = int(match[1]), int(match[2])
            now = time.monotonic()
            if epoch > row["epoch"]:
                previous = row["last_epoch_at"]
                if previous is not None:
                    row["epoch_durations"].append(now - previous)
                row["last_epoch_at"] = now
            loss, accuracy, val_loss, val_accuracy = map(float, match.groups()[2:])
            row.update(
                epoch=epoch,
                epochs=total,
                train_loss=loss,
                train_acc=accuracy,
                val_loss=val_loss,
                val_acc=val_accuracy,
                batch=None,
                batch_total=None,
                detail=f"Đã xong epoch {epoch}/{total}",
                stage="train",
            )
            row["best_val"] = (
                val_loss if row["best_val"] is None else min(row["best_val"], val_loss)
            )
        elif mode == "train":
            batch = BATCH_LOG.search(line)
            if batch:
                row.update(
                    stage="train" if batch[1] == "Training" else "validation",
                    batch=int(batch[2]),
                    batch_total=int(batch[3]),
                    detail=f"Epoch {min(row['epoch'] + 1, row['epochs'])}/{row['epochs']}",
                )

    def render(self, force=False):
        now = time.monotonic()
        if not force and now - self.last_render < 1:
            return
        self.last_render = now
        rows_html = []
        for name, row in self.rows.items():
            color = (
                "#dcfce7"
                if row["stage"] == "done"
                else "#fee2e2"
                if row["stage"] in ("error", "stopped")
                else "#e0f2fe"
                if name == self.active
                else "#f1f5f9"
            )
            cells = [
                name.upper(),
                self.LABELS[row["stage"]],
                f"{row['epoch']}/{row['epochs']}",
                number(row["train_loss"]),
                number(row["val_loss"]),
                percent(row["train_acc"]),
                percent(row["val_acc"]),
                percent(row["test_acc"]),
                number(row["test_f1"]),
                duration(row["train_seconds"]),
            ]
            rows_html.append(
                '<tr style="background:'
                + color
                + '">'
                + "".join("<td>" + html.escape(str(v)) + "</td>" for v in cells)
                + "</tr>"
            )
        completed = sum(r["stage"] == "done" for r in self.rows.values())
        epochs_done = sum(r["epoch"] for r in self.rows.values())
        epochs_total = sum(r["epochs"] for r in self.rows.values())
        active = self.rows.get(self.active)
        active_html = "<p>Chọn ô model bên dưới để bắt đầu.</p>"
        if active and self.active is not None:
            eta = "Đang ước lượng; cần ít nhất 2 epoch hoàn tất"
            if active["stage"] in ("train", "validation") and active["epoch_durations"]:
                recent = active["epoch_durations"][-5:]
                eta = (
                    "~"
                    + duration(
                        sum(recent) / len(recent) * max(0, active["epochs"] - active["epoch"])
                    )
                    + " còn lại cho train model này"
                )
            elif active["stage"] not in ("train", "validation"):
                eta = "—"
            batch_html = ""
            if active["batch_total"]:
                batch_html = f'<div>Batch: {active["batch"]}/{active["batch_total"]}</div><progress value="{active["batch"]}" max="{active["batch_total"]}"></progress>'
            active_html = f'<h3>{html.escape(self.active.upper())} · {self.LABELS[active["stage"]]}</h3><div>{html.escape(active["detail"])}</div><div>Epoch hoàn tất: {active["epoch"]}/{active["epochs"]} · Best val loss: {number(active["best_val"])}</div><progress value="{active["epoch"]}" max="{active["epochs"]}"></progress>{batch_html}<div>Thời gian bước hiện tại: {duration(now - self.stage_started)} · ETA: {html.escape(eta)}</div>'
        age = (
            "Chưa có log từ subprocess"
            if self.last_output is None
            else f"Log gần nhất: {int(now - self.last_output)} giây trước"
        )
        log_html = html.escape("\n".join(self.tail)) or "Đang chờ dữ liệu / tiến trình khởi tạo…"
        console_html = html.escape(str(self.console_path or run_dir / "console"))
        content = f'''<div class="a1-monitor"><style>
.a1-monitor{{font-family:Arial,sans-serif;color:#0f172a;background:#fff;padding:18px;border:1px solid #cbd5e1;border-radius:12px;line-height:1.6}}
.a1-monitor h2,.a1-monitor h3{{margin:4px 0 10px;color:#0f172a}}.a1-monitor progress{{width:100%;height:18px;accent-color:#0284c7}}
.a1-monitor table{{width:100%;border-collapse:collapse;font-size:12px;color:#0f172a}}.a1-monitor th,.a1-monitor td{{padding:8px;border:1px solid #cbd5e1;text-align:left;white-space:nowrap}}
.a1-monitor th{{background:#0f172a;color:#fff}}.a1-monitor pre{{background:#0f172a;color:#e2e8f0;padding:12px;white-space:pre-wrap;overflow-wrap:anywhere;font-size:11px;max-height:160px;overflow:auto}}
.a1-monitor .card{{background:#f0f9ff;padding:12px;border-radius:8px;margin:12px 0}}.a1-monitor small{{color:#475569}}
</style><h2>A1 · 6 mô hình Fashion-MNIST</h2><div>GPU: <b>{html.escape(str(environment["gpu"]))}</b> · Theo dõi được {duration(now - self.started)}</div>
<div>Model hoàn tất (train + test + sao lưu): <b>{completed}/{len(models)}</b> · Epoch train toàn bộ: <b>{epochs_done}/{epochs_total}</b></div>
<progress value="{completed}" max="{len(models)}"></progress><div class="card">{active_html}</div>
<div style="overflow-x:auto"><table><thead><tr><th>Model</th><th>Trạng thái</th><th>Epoch đã xong</th><th>Train loss</th><th>Val loss</th><th>Train acc</th><th>Val acc</th><th>Test acc</th><th>Test F1</th><th>Train đã đo</th></tr></thead><tbody>{"".join(rows_html)}</tbody></table></div>
<small>Loss/accuracy cập nhật sau mỗi epoch hoàn tất. ETA chỉ ước lượng train của model hiện tại, chưa gồm test và sao lưu. {html.escape(age)}.</small>
<pre>{log_html}</pre><small>Log đầy đủ: {console_html}<br>Drive: {html.escape(str(saved_dir))}</small></div>'''
        clear_output(wait=True)
        display(HTML(content))


def run_logged(model_name, mode, module):
    console_dir = run_dir / "console"
    console_dir.mkdir(exist_ok=True)
    log_path = console_dir / f"{model_name}_{mode}.log"
    monitor.console_path = log_path
    monitor.tail.clear()
    monitor.last_output = None
    monitor.stage(
        model_name,
        mode,
        "Khởi tạo / tải dữ liệu" if mode == "train" else "Đánh giá 10.000 ảnh và xuất hình",
    )
    child_env = os.environ.copy()
    child_env.update(PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8", TQDM_MININTERVAL="2")
    child_env.pop("TQDM_DISABLE", None)
    command = [
        sys.executable,
        "-u",
        "-m",
        module,
        "--data",
        "fashion_mnist",
        "--config",
        str(config_dir / f"{model_name}.yaml"),
    ]
    process = subprocess.Popen(
        command,
        cwd=repo_dir,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
        env=child_env,
    )
    output_stream = process.stdout
    assert output_stream is not None
    events = queue.Queue()
    reader_done = threading.Event()

    def read_output():
        try:
            for line in output_stream:
                events.put(line)
        finally:
            reader_done.set()

    reader = threading.Thread(target=read_output, daemon=True)
    reader.start()
    try:
        with log_path.open("w", encoding="utf-8") as log:
            while not reader_done.is_set() or not events.empty():
                try:
                    line = events.get(timeout=0.25)
                    log.write(line)
                    log.flush()
                    monitor.consume(model_name, line, mode)
                except queue.Empty:
                    pass
                monitor.render()
        return_code = process.wait()
        if return_code:
            raise RuntimeError(
                f"{model_name} {mode} lỗi (exit {return_code}). Xem {log_path}\n"
                + "\n".join(monitor.tail)
            )
    finally:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        reader.join(timeout=2)
        output_stream.close()
        monitor.render(force=True)


def run_one_model(model_name):
    assert model_name in models
    assert torch.cuda.is_available(), "CUDA không khả dụng."
    assert torch.cuda.get_device_name(0) == environment["gpu"], "GPU đã thay đổi."
    row = monitor.rows[model_name]
    needs_train = not training_complete(model_name)
    try:
        if needs_train:
            row.update(
                epoch=0,
                epoch_durations=[],
                last_epoch_at=None,
                best_val=None,
                train_loss=None,
                val_loss=None,
                train_acc=None,
                val_acc=None,
                test_acc=None,
                test_f1=None,
                train_seconds=None,
            )
            run_logged(model_name, "train", "scripts.training.train_image_classification")
            assert training_complete(model_name), "Train chưa tạo đủ summary/checkpoint."
        monitor.apply_summary(model_name)
        monitor.stage(model_name, "backup", "Lưu checkpoint và kết quả train vào Drive")
        backup_outputs()
        if needs_train or not evaluation_complete(model_name):
            run_logged(model_name, "evaluate", "scripts.evaluation.evaluate")
        assert evaluation_complete(model_name), "Kết quả test không khớp cấu hình/checkpoint."
        record = read_json(run_dir / "results" / configs[model_name]["name"] / "test_result.json")
        row.update(test_acc=record["metrics"]["accuracy"], test_f1=record["metrics"]["macro_f1"])
        monitor.stage(model_name, "backup", "Lưu kết quả test và hình phân tích vào Drive")
        backup_outputs()
        monitor.stage(model_name, "done", "Đã hoàn tất train, test và sao lưu Drive")
    except KeyboardInterrupt:
        monitor.stage(
            model_name,
            "stopped",
            "Đã dừng; model đang train cần chạy lại từ đầu. Model hoàn tất được giữ nguyên.",
        )
        raise
    except Exception as exc:
        monitor.tail.append(str(exc))
        monitor.stage(model_name, "error", str(exc))
        raise


monitor = A1Progress()
print("Đã sẵn sàng. Xem bảng tiến trình ngay dưới ô model đang chạy ở phần 3.1–3.6.")

### 3.1. LINEAR — model 1/6
Ô này tự train, test và sao lưu model. Nếu đã có kết quả hoàn chỉnh khớp cấu hình, bước đó được bỏ qua.


In [ ]:
run_one_model("linear")

### 3.2. MLP — model 2/6
Ô này tự train, test và sao lưu model. Nếu đã có kết quả hoàn chỉnh khớp cấu hình, bước đó được bỏ qua.


In [ ]:
run_one_model("mlp")

### 3.3. CNN — model 3/6
Ô này tự train, test và sao lưu model. Nếu đã có kết quả hoàn chỉnh khớp cấu hình, bước đó được bỏ qua.


In [ ]:
run_one_model("cnn")

### 3.4. GRU — model 4/6
Ô này tự train, test và sao lưu model. Nếu đã có kết quả hoàn chỉnh khớp cấu hình, bước đó được bỏ qua.


In [ ]:
run_one_model("gru")

### 3.5. LSTM — model 5/6
Ô này tự train, test và sao lưu model. Nếu đã có kết quả hoàn chỉnh khớp cấu hình, bước đó được bỏ qua.


In [ ]:
run_one_model("lstm")

### 3.6. TRANSFORMER — model 6/6
Ô này tự train, test và sao lưu model. Nếu đã có kết quả hoàn chỉnh khớp cấu hình, bước đó được bỏ qua.


In [ ]:
run_one_model("transformer")

## 4. So sánh và xuất ZIP
Bảng chỉ dùng 6 kết quả mới trong đợt Colab này, không trộn bản CPU hoặc bản nháp cũ.


In [ ]:
import zipfile

for model_name in models:
    result_path = run_dir / "results" / configs[model_name]["name"] / "test_result.json"
    assert result_path.exists(), f"Chưa có kết quả: {model_name}"

subprocess.run(
    [
        sys.executable,
        "-m",
        "scripts.evaluation.compare",
        "--results-dir",
        str(run_dir / "results"),
    ],
    cwd=repo_dir,
    check=True,
)
backup_outputs()
zip_path = str(saved_dir / "a1_cuda_outputs.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for source, relative in output_files():
        archive.write(source, arcname=relative.as_posix())
print("ZIP trên Drive:", zip_path)
print((run_dir / "results/comparison_test.md").read_text())

## 5. Tải kết quả về máy (tùy chọn)
ZIP đã có trong Drive. Ô dưới tải thêm một bản về máy bạn. Giữ thư mục đợt CUDA riêng; chưa ghi đè bộ checkpoint/results CPU cũ.


In [ ]:
from google.colab import files  # ty: ignore[unresolved-import]

files.download(zip_path)